In [ ]:
# Notebook: Routh Table Construction & Stability Analysis (Example)
import numpy as np
import control as ct

# 1. Define polynomial coefficients: P(s) = s^6 + 4s^5 + 3s^4 + 2s^3 + s^2 + 4s + 4
# Coefficients ordered from highest to lowest power of s
coeffs = [1, 4, 3, 2, 1, 4, 4]
N = len(coeffs) - 1

print("=== Routh-Hurwitz Array Analysis ===")
print(f"Polynomial Degree N = {N}")
print(f"Coefficients: {coeffs}")
print("-" * 50)

# 2. Automated Routh table construction function
def routh_table(poly_coeffs):
    n = len(poly_coeffs) - 1
    # Initialize Routh array with zeros
    rows = n + 1
    cols = (n // 2) + 1
    R = np.zeros((rows, cols))
    
    # Fill the first two rows
    R[0, :len(poly_coeffs[0::2])] = poly_coeffs[0::2]
    R[1, :len(poly_coeffs[1::2])] = poly_coeffs[1::2]
    
    # Compute subsequent rows
    for i in range(2, rows):
        for j in range(cols - 1):
            if R[i-1, 0] == 0:
                # Handle zero-row special case practically by adding a small epsilon
                R[i-1, 0] = 1e-6
            # Routh determinant formula
            num = -(R[i-2, 0] * R[i-1, j+1] - R[i-2, j+1] * R[i-1, 0])
            den = R[i-1, 0]
            R[i, j] = num / den
            
    return R

R_table = routh_table(coeffs)

# Print the Routh array with row labels from s^N down to s^0 (clean float values)
print("• Routh Table:")
for i in range(N + 1):
    row_power = N - i
    row_vals = [round(float(val), 3) for val in R_table[i]]
    print(f"  s^{row_power:<2} | {row_vals}")

print("-" * 50)

# 3. Count sign changes in the first column to determine unstable roots
first_col = R_table[:, 0]
sign_changes = 0
for i in range(len(first_col) - 1):
    if first_col[i] * first_col[i+1] < 0:
        sign_changes += 1

print(f"• Number of Sign Changes in First Column: {sign_changes}")
print(f"• Stability Status: {'Stable' if sign_changes == 0 else f'Unstable ({sign_changes} roots with positive real parts)'}")

# 4. System poles verification
num = [1]
den = coeffs
sys = ct.tf(num, den)
poles = ct.poles(sys)
print("-" * 50)
print(f"• Exact System Poles:")
for p in np.round(poles, 4):
    print(f"  p = {p}")